# Two-Tower Retrieval Experiments

Stage-1 retrieval training with AWS Managed MLflow + Optuna HPO.

Spec: `docs/superpowers/specs/2026-06-11-two-tower-retrieval-experiments-design.md`


## 1. Setup

### 1.1 Environment & credentials

Load `.env.local` (gitignored) for AWS credentials and MLflow/Optuna endpoints.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

for _nb in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (_nb / "utils").is_dir():
        sys.path[:0] = [str(_nb)]
        break

from utils.config_loader import find_repo_root
from utils.ml_config import MLInfraConfig

REPO_ROOT = find_repo_root()
load_dotenv(REPO_ROOT / ".env.local")
load_dotenv(REPO_ROOT / ".env")

ml_cfg = MLInfraConfig.from_env()
ml_cfg.validate_for_aws()
print("S3 bucket:", ml_cfg.s3_bucket)
print("MLflow URI set:", bool(ml_cfg.mlflow_tracking_uri))


### 1.2 Load configs (YAML + .env)


In [ ]:
from utils.config_loader import find_repo_root, load_feature_engineering_config
from utils.two_tower_training_helpers import load_two_tower_yaml, load_search_space_yaml

repo_root = find_repo_root()
fe_cfg = load_feature_engineering_config(repo_root=repo_root, environment="aws")
tt_cfg = load_two_tower_yaml(repo_root)
search_space = load_search_space_yaml(repo_root)
print("Default embedding_dim:", tt_cfg["embedding_dim"])
print("Optuna n_trials:", tt_cfg["optuna"]["n_trials"])


### 1.3 Verify feature data schema


In [ ]:
import pandas as pd
from utils.two_tower_training_helpers import verify_schema

features_path = (
    repo_root / fe_cfg["local_s3_root"] / "dataset" / fe_cfg["dataset_name"] / "features" / "transactions"
)
df = pd.read_parquet(features_path)
verify_schema(df)
print("Loaded rows:", len(df))
df[["customer_id", "article_id", "t_dat", "item_category", "txn_month_sin", "txn_month_cos"]].head()


## 2. Infrastructure checks

### 2.1 MLflow tracking server status

Start/stop the server between experiment sessions.


In [ ]:
from utils.two_tower_training_helpers import mlflow_server_status

status = mlflow_server_status(ml_cfg.mlflow_tracking_server_name, ml_cfg.aws_region)
print("MLflow server status:", status)


### 2.2 Optuna RDS connectivity


In [ ]:
import optuna

study = optuna.create_study(
    study_name=tt_cfg["optuna"]["study_name"],
    storage=ml_cfg.optuna_storage_uri,
    load_if_exists=True,
    direction=tt_cfg["optuna"]["direction"],
)
print("Study loaded:", study.study_name, "trials=", len(study.trials))


## 3. Data preparation

### 3.1 Load transactions features

### 3.2 Apply temporal split & stage to S3


In [ ]:
from utils.two_tower_training_helpers import apply_temporal_split, new_run_id, stage_splits_s3

temporal = tt_cfg["temporal_split"]
train_df, val_df, test_df = apply_temporal_split(df, temporal)
print("train", len(train_df), "val", len(val_df), "test", len(test_df))

run_id = new_run_id()
split_uris = stage_splits_s3(
    train_df, val_df, test_df,
    bucket=ml_cfg.s3_bucket,
    run_id=run_id,
    region=ml_cfg.aws_region,
)
split_uris


## 4. Baseline training (optional smoke)

### 4.1 Launch single Training Job with guide defaults


In [ ]:
# Optional smoke: use pipelines/sagemaker/launch_training_job.py with guide defaults.


## 5. Hyperparameter optimization

### 5.1 Configure search space (n_trials=3)

### 5.2 Launch Processing orchestrator job


In [ ]:
import subprocess

os.environ["MLFLOW_TRACKING_URI"] = ml_cfg.mlflow_tracking_uri
os.environ["OPTUNA_STORAGE_URI"] = ml_cfg.optuna_storage_uri
os.environ["SAGEMAKER_ROLE_ARN"] = ml_cfg.sagemaker_role_arn
os.environ["S3_BUCKET"] = ml_cfg.s3_bucket
os.environ["FEATURE_SNAPSHOT"] = split_uris["train"]

proc_cmd = [
    sys.executable,
    str(repo_root / "pipelines" / "sagemaker" / "hpo_processing_job.py"),
    "--train-uri", split_uris["train"],
    "--val-uri", split_uris["val"],
]
print(" ".join(proc_cmd))
# subprocess.run(proc_cmd, check=True)


### 5.3 Monitor trials in MLflow


In [ ]:
import mlflow

mlflow.set_tracking_uri(ml_cfg.mlflow_tracking_uri)
mlflow.set_experiment(ml_cfg.mlflow_experiment)
runs = mlflow.search_runs(filter_string="tags.model = 'two_tower'", max_results=10)
runs


## 6. Final evaluation

### 6.1 Select best trial

### 6.2 Launch test eval Training Job


In [ ]:
best_params = study.best_params
best_params


In [ ]:
# Final test eval with frozen best params (informational only)
# Uncomment and adapt launch_training_job import when running on AWS.


## 7. Results & handoff

### 7.1 Compare runs / plots

### 7.2 Export best params


In [ ]:
from utils.two_tower_training_helpers import export_best_params

if study.best_params:
    out_path = export_best_params(study.best_params, repo_root)
    print("Updated", out_path)
